# Avaliação de Recuperação de Informação — all-MiniLM-L12-v2

Este notebook avalia o desempenho do modelo **all-MiniLM-L12-v2** em tarefas de **Recuperação de Informação (Information Retrieval — IR)**.

O domínio escolhido é a **Educação de Jovens e Adultos (EJA)**, tema relevante para sistemas de busca semântica em contextos educacionais brasileiros.

**Métricas avaliadas:**
- Recall@K (K = 1, 3, 5, 10)
- Precision@K (K = 1, 3, 5, 10)
- MRR@10 (Mean Reciprocal Rank)
- nDCG@10 (Normalized Discounted Cumulative Gain)

---

## Seção 1 — Configuração do Ambiente

Importação das bibliotecas necessárias e carregamento do modelo de embeddings.

In [ ]:
# ─── Importações ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

print('✅ Bibliotecas importadas com sucesso.')

In [ ]:
# ─── Caminhos base ───────────────────────────────────────────────────────────
BASE_DIR    = Path('../')
DATA_DIR    = BASE_DIR / 'data' / 'test_sets'
PLOTS_DIR   = BASE_DIR / 'results' / 'plots'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'

# Garantir que os diretórios existam
for d in [DATA_DIR, PLOTS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('📁 Diretórios verificados:')
for d in [DATA_DIR, PLOTS_DIR, METRICS_DIR]:
    print(f'   {d.resolve()}')

In [ ]:
# ─── Carregamento do modelo ───────────────────────────────────────────────────
MODEL_NAME = 'sentence-transformers/all-MiniLM-L12-v2'

print(f'⏳ Carregando modelo: {MODEL_NAME} ...')
t0 = time.time()

try:
    model = SentenceTransformer(MODEL_NAME)
    t1 = time.time()
    print(f'✅ Modelo carregado em {t1 - t0:.2f}s')
    print(f'   Dimensão dos embeddings: {model.get_sentence_embedding_dimension()}')
    print(f'   Comprimento máximo de tokens: {model.max_seq_length}')
except Exception as e:
    print(f'❌ Erro ao carregar modelo: {e}')
    raise

---
## Seção 2 — Corpus de Documentos

Criamos um corpus com **30 documentos** sobre diferentes aspectos da Educação de Jovens e Adultos (EJA).
Cada documento possui um identificador único (`doc_id`) e um texto representativo do subtema.

In [ ]:
# ─── Definição do corpus (30 documentos sobre EJA) ───────────────────────────
corpus = {
    "doc_01": (
        "A Educação de Jovens e Adultos (EJA) é uma modalidade da educação básica "
        "destinada àqueles que não tiveram acesso ou continuidade de estudos no "
        "ensino fundamental e médio na idade própria. É um direito garantido pela "
        "Constituição Federal e pela Lei de Diretrizes e Bases da Educação Nacional."
    ),
    "doc_02": (
        "A matrícula na EJA pode ser realizada em qualquer época do ano, conforme "
        "disponibilidade de vagas nas unidades escolares. O aluno deve comparecer à "
        "secretaria da escola com documentos de identificação, comprovante de "
        "residência e histórico escolar, quando disponível."
    ),
    "doc_03": (
        "A avaliação na EJA deve ser contínua, cumulativa e diagnóstica, respeitando "
        "o ritmo de aprendizagem de cada estudante. Os instrumentos de avaliação "
        "incluem provas escritas, trabalhos, seminários e portfólios, valorizando "
        "os conhecimentos prévios dos educandos adquiridos ao longo da vida."
    ),
    "doc_04": (
        "Para efetuar a matrícula na EJA, o estudante deve apresentar: documento de "
        "identidade com foto (RG, CNH ou passaporte), CPF, comprovante de residência "
        "atualizado e, se disponível, histórico escolar ou declaração de escolaridade "
        "anterior. Menores de 15 anos não podem se matricular no ensino fundamental EJA."
    ),
    "doc_05": (
        "As aulas da EJA geralmente funcionam no período noturno, para atender "
        "trabalhadores que não podem estudar durante o dia. Algumas redes também "
        "oferecem turmas no período matutino e vespertino. A carga horária mínima "
        "é de 1.600 horas para o ensino fundamental e 1.200 horas para o ensino médio."
    ),
    "doc_06": (
        "A EJA é regulamentada pela Lei de Diretrizes e Bases da Educação Nacional "
        "(LDB — Lei nº 9.394/1996), especialmente nos artigos 37 e 38, que estabelecem "
        "os princípios, a organização e as diretrizes curriculares nacionais para "
        "essa modalidade educacional. As Diretrizes Curriculares Nacionais para a EJA "
        "foram definidas pelo Parecer CNE/CEB nº 11/2000."
    ),
    "doc_07": (
        "O público-alvo da EJA são jovens com 15 anos ou mais para o ensino fundamental "
        "e 18 anos ou mais para o ensino médio, que não concluíram a educação básica "
        "na idade regular. A modalidade atende principalmente trabalhadores, donas de "
        "casa, pessoas privadas de liberdade e idosos que desejam retomar seus estudos."
    ),
    "doc_08": (
        "A EJA pode ser ofertada nas modalidades presencial, semipresencial e, mais "
        "recentemente, a distância (EaD). A modalidade presencial exige frequência "
        "regular às aulas; a semipresencial combina momentos presenciais com "
        "atividades autônomas; e a EaD permite o estudo totalmente remoto, com "
        "encontros periódicos para avaliação."
    ),
    "doc_09": (
        "A EJA tem raízes históricas no Brasil que remontam ao período colonial, "
        "com missões jesuíticas que alfabetizavam indígenas. No século XX, destacam-se "
        "as campanhas de alfabetização da década de 1940, o Método Paulo Freire nos "
        "anos 1960 e, após a redemocratização, a criação do MOBRAL e, posteriormente, "
        "do FUNDEF e FUNDEB, que passaram a financiar a EJA."
    ),
    "doc_10": (
        "A certificação na EJA pode ocorrer por conclusão de curso presencial ou pelo "
        "Exame Nacional para Certificação de Competências de Jovens e Adultos (ENCCEJA), "
        "que avalia competências equivalentes ao ensino fundamental e médio. O certificado "
        "tem validade nacional e é reconhecido por empregadores e instituições de ensino superior."
    ),
    "doc_11": (
        "O estudante da EJA pode solicitar transferência entre escolas ou redes de ensino "
        "a qualquer momento, mediante apresentação do histórico escolar ou declaração de "
        "frequência e notas emitida pela escola de origem. A escola receptora deve acolher "
        "o aluno e aproveitar os estudos já realizados, sem exigir repetição de conteúdos "
        "já certificados."
    ),
    "doc_12": (
        "A frequência mínima na EJA presencial é de 75% da carga horária total do curso, "
        "assim como no ensino regular. Faltas excessivas podem levar à reprovação por "
        "frequência. Algumas redes preveem abono de faltas para trabalhadores com "
        "comprovante de serviço em horário de aula."
    ),
    "doc_13": (
        "O currículo da EJA deve ser contextualizado à realidade dos educandos adultos, "
        "valorizando suas experiências de vida e trabalho. As diretrizes curriculares "
        "preveem o desenvolvimento de competências em linguagens, matemática, ciências "
        "da natureza, ciências humanas e formação profissional, de forma integrada e "
        "significativa para o mundo do trabalho."
    ),
    "doc_14": (
        "O material didático para a EJA deve ser adaptado às características e "
        "necessidades dos estudantes adultos, com linguagem acessível, exemplos "
        "práticos e conteúdos relacionados ao cotidiano. O Programa Nacional do Livro "
        "Didático (PNLD-EJA) distribui gratuitamente livros específicos para essa "
        "modalidade às escolas públicas."
    ),
    "doc_15": (
        "Os professores da EJA devem ter formação em licenciatura e, preferencialmente, "
        "especialização em educação de adultos. A prática pedagógica na EJA exige "
        "sensibilidade para lidar com a diversidade de perfis dos estudantes, "
        "metodologias ativas e capacidade de contextualizar o conteúdo à realidade "
        "do trabalhador adulto."
    ),
    "doc_16": (
        "As escolas que ofertam EJA devem dispor de infraestrutura adequada para "
        "receber estudantes adultos: salas de aula confortáveis, iluminação adequada "
        "para o período noturno, banheiros acessíveis, laboratórios de informática e "
        "biblioteca com acervo atualizado. A acessibilidade para pessoas com deficiência "
        "é requisito legal."
    ),
    "doc_17": (
        "A acessibilidade na EJA deve garantir o acesso de pessoas com deficiência "
        "física, visual, auditiva ou intelectual. Isso inclui rampas de acesso, "
        "intérpretes de Libras, materiais em Braille, tecnologias assistivas e "
        "adaptações curriculares. A Lei Brasileira de Inclusão (Lei nº 13.146/2015) "
        "reforça esses direitos."
    ),
    "doc_18": (
        "O uso de tecnologias digitais na EJA tem crescido significativamente, "
        "especialmente após a pandemia de COVID-19. Plataformas de ensino online, "
        "aplicativos educacionais e videoaulas são ferramentas que ampliam o acesso "
        "ao conhecimento. No entanto, a exclusão digital ainda é um desafio para "
        "parte do público da EJA."
    ),
    "doc_19": (
        "A EJA conta com parcerias entre o poder público e organizações da sociedade "
        "civil, empresas e movimentos sociais. O Programa Brasil Alfabetizado, o SESI, "
        "o SENAI e organizações não governamentais complementam a oferta pública, "
        "especialmente em regiões de difícil acesso ou com alta demanda."
    ),
    "doc_20": (
        "O financiamento da EJA é assegurado pelo Fundo de Manutenção e Desenvolvimento "
        "da Educação Básica (FUNDEB), que destina recursos específicos para essa "
        "modalidade. O fator de ponderação do FUNDEB para a EJA é menor do que para "
        "outras modalidades, o que é apontado como um dos desafios para sua expansão "
        "e melhoria da qualidade."
    ),
    "doc_21": (
        "Os principais desafios da EJA incluem: alta taxa de evasão escolar, "
        "financiamento insuficiente, falta de formação especializada para docentes, "
        "infraestrutura precária em muitas escolas, estigma social associado ao "
        "atraso escolar e dificuldade de conciliar estudo com trabalho e obrigações "
        "familiares."
    ),
    "doc_22": (
        "Entre as oportunidades para a EJA, destacam-se: a crescente demanda por "
        "qualificação profissional, o avanço das tecnologias de ensino a distância, "
        "políticas de valorização da educação continuada, integração com o mundo "
        "do trabalho por meio da EJA integrada à educação profissional e o potencial "
        "transformador da educação para redução da desigualdade social."
    ),
    "doc_23": (
        "Exemplos de sucesso na EJA incluem escolas que adotaram metodologias "
        "inovadoras, como a pedagogia de projetos, o ensino por competências e "
        "a integração com o mundo do trabalho. Redes municipais como as de São Paulo "
        "e Recife desenvolveram programas premiados que reduziram a evasão e "
        "melhoraram o desempenho dos estudantes."
    ),
    "doc_24": (
        "Em comparação com o ensino regular, a EJA possui carga horária reduzida, "
        "currículo flexível e metodologia diferenciada. Enquanto o ensino regular "
        "é organizado em séries anuais, a EJA pode ser organizada em semestres ou "
        "módulos. A avaliação na EJA tende a ser mais flexível, reconhecendo os "
        "saberes adquiridos fora da escola."
    ),
    "doc_25": (
        "A relação entre EJA e o mundo do trabalho é central para essa modalidade. "
        "Muitos estudantes buscam a EJA para melhorar suas condições de emprego, "
        "conquistar promoções ou mudar de área profissional. A EJA integrada à "
        "educação profissional permite ao aluno concluir o ensino básico e obter "
        "qualificação técnica simultaneamente."
    ),
    "doc_26": (
        "A EJA contribui para o exercício pleno da cidadania, pois a escolaridade "
        "é condição fundamental para participação política, social e cultural. "
        "Estudantes da EJA desenvolvem maior autonomia para ler contratos, "
        "compreender direitos trabalhistas, exercer o voto consciente e participar "
        "de decisões comunitárias."
    ),
    "doc_27": (
        "A EJA e a saúde são temas interligados: a escolaridade influencia "
        "diretamente os hábitos de saúde, o acesso a serviços médicos e a "
        "compreensão de informações sanitárias. Programas de EJA integrados à "
        "saúde ensinam prevenção de doenças, higiene, nutrição e direitos dos "
        "pacientes, contribuindo para melhor qualidade de vida."
    ),
    "doc_28": (
        "A alfabetização de adultos é a etapa inicial da EJA, voltada para pessoas "
        "que não sabem ler nem escrever. O Método Paulo Freire, baseado em palavras "
        "geradoras do universo cultural do educando, é referência mundial nessa área. "
        "O Brasil ainda possui milhões de adultos analfabetos, especialmente nas "
        "regiões Norte e Nordeste."
    ),
    "doc_29": (
        "A EJA integrada à educação profissional (EJA-EP) é uma modalidade que "
        "articula a conclusão da educação básica com a formação técnica. O estudante "
        "cursa disciplinas do ensino fundamental ou médio e, ao mesmo tempo, se "
        "qualifica em uma área profissional como informática, administração, "
        "enfermagem ou construção civil."
    ),
    "doc_30": (
        "As perspectivas para a EJA no Brasil apontam para a expansão do ensino "
        "híbrido, maior integração com o mercado de trabalho e uso intensivo de "
        "tecnologias educacionais. Políticas como o novo FUNDEB e o Plano Nacional "
        "de Educação (PNE) preveem metas específicas para redução do analfabetismo "
        "e aumento da escolaridade de adultos no Brasil."
    ),
}

print(f'📚 Corpus criado com {len(corpus)} documentos.')
print()
print('Primeiros 3 documentos:')
for doc_id, text in list(corpus.items())[:3]:
    preview = text[:120].replace('\n', ' ')
    print(f'  [{doc_id}] {preview}...')

In [ ]:
# ─── Salvar corpus em JSON ────────────────────────────────────────────────────
corpus_path = DATA_DIR / 'corpus.json'

try:
    with open(corpus_path, 'w', encoding='utf-8') as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)
    print(f'✅ Corpus salvo em: {corpus_path.resolve()}')
    print(f'   Tamanho do arquivo: {corpus_path.stat().st_size / 1024:.1f} KB')
except Exception as e:
    print(f'❌ Erro ao salvar corpus: {e}')
    raise

---
## Seção 3 — Consultas e Relevância

Definimos **20 consultas** relacionadas ao tema EJA e mapeamos cada consulta aos documentos relevantes do corpus.
A relevância é definida manualmente com base no conteúdo semântico de cada documento.

In [ ]:
# ─── Definição das consultas (20 perguntas sobre EJA) ────────────────────────
queries = {
    "q_01": "O que é a Educação de Jovens e Adultos?",
    "q_02": "Como fazer matrícula na EJA?",
    "q_03": "Quais documentos são necessários para se inscrever na EJA?",
    "q_04": "Como funciona a avaliação dos alunos na EJA?",
    "q_05": "Quais são os horários das aulas da EJA?",
    "q_06": "Qual é a legislação que regulamenta a EJA no Brasil?",
    "q_07": "Quem pode estudar na EJA?",
    "q_08": "A EJA pode ser feita a distância?",
    "q_09": "Qual é a história da educação de adultos no Brasil?",
    "q_10": "Como obter certificado pela EJA ou ENCCEJA?",
    "q_11": "Como fazer transferência entre escolas na EJA?",
    "q_12": "Qual é a frequência mínima exigida na EJA?",
    "q_13": "Quais são os desafios enfrentados pela EJA no Brasil?",
    "q_14": "Como a EJA contribui para o mercado de trabalho?",
    "q_15": "A EJA oferece formação profissional técnica?",
    "q_16": "Como a EJA apoia o exercício da cidadania?",
    "q_17": "Qual a relação entre EJA e saúde pública?",
    "q_18": "Como é feita a alfabetização de adultos?",
    "q_19": "Como é o financiamento da EJA?",
    "q_20": "Quais são as perspectivas futuras para a EJA no Brasil?",
}

print(f'🔎 Total de consultas criadas: {len(queries)}')
print()
for qid, qtxt in queries.items():
    print(f'  [{qid}] {qtxt}')

In [ ]:
# ─── Mapeamento de relevância (query_id -> set de doc_ids relevantes) ─────────
# Cada consulta tem 1 a 3 documentos relevantes definidos manualmente
relevant_docs = {
    "q_01": {"doc_01", "doc_07"},          # conceito geral + público-alvo
    "q_02": {"doc_02", "doc_04"},          # matrícula + documentação
    "q_03": {"doc_04", "doc_02"},          # documentação + matrícula
    "q_04": {"doc_03"},                    # avaliação
    "q_05": {"doc_05"},                    # funcionamento/horários
    "q_06": {"doc_06"},                    # legislação
    "q_07": {"doc_07", "doc_01"},          # público-alvo + conceito geral
    "q_08": {"doc_08"},                    # modalidades (EaD)
    "q_09": {"doc_09"},                    # histórico
    "q_10": {"doc_10"},                    # certificação
    "q_11": {"doc_11"},                    # transferência
    "q_12": {"doc_12"},                    # frequência
    "q_13": {"doc_21", "doc_20"},          # desafios + financiamento
    "q_14": {"doc_25", "doc_22"},          # EJA e trabalho + oportunidades
    "q_15": {"doc_29", "doc_25"},          # EJA integrada + trabalho
    "q_16": {"doc_26"},                    # EJA e cidadania
    "q_17": {"doc_27"},                    # EJA e saúde
    "q_18": {"doc_28"},                    # alfabetização
    "q_19": {"doc_20"},                    # financiamento
    "q_20": {"doc_30", "doc_22"},          # perspectivas + oportunidades
}

print('📌 Mapeamento de relevância (consulta → documentos relevantes):')
for qid, docs in relevant_docs.items():
    print(f'  [{qid}] → {sorted(docs)}')

In [ ]:
# ─── Serializar relevância para JSON (sets → lists) ───────────────────────────
relevant_docs_serializable = {
    qid: sorted(list(docs)) for qid, docs in relevant_docs.items()
}

queries_path   = DATA_DIR / 'queries.json'
relevant_path  = DATA_DIR / 'relevant_docs.json'

try:
    with open(queries_path, 'w', encoding='utf-8') as f:
        json.dump(queries, f, ensure_ascii=False, indent=2)
    print(f'✅ Consultas salvas em: {queries_path.resolve()}')

    with open(relevant_path, 'w', encoding='utf-8') as f:
        json.dump(relevant_docs_serializable, f, ensure_ascii=False, indent=2)
    print(f'✅ Relevâncias salvas em: {relevant_path.resolve()}')
except Exception as e:
    print(f'❌ Erro ao salvar arquivos: {e}')
    raise

---
## Seção 4 — Codificação (Embeddings)

Codificamos todos os documentos do corpus e todas as consultas utilizando o modelo **all-MiniLM-L12-v2**.
Os embeddings resultantes são vetores de 384 dimensões que representam o significado semântico dos textos.

In [ ]:
# ─── Codificação dos documentos do corpus ─────────────────────────────────────
doc_ids   = list(corpus.keys())
doc_texts = list(corpus.values())

print(f'⏳ Codificando {len(doc_texts)} documentos...')
t0 = time.time()

try:
    doc_embeddings = model.encode(
        doc_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    t1 = time.time()
    print(f'\n✅ Documentos codificados em {t1 - t0:.2f}s')
    print(f'   Shape dos embeddings de documentos: {doc_embeddings.shape}')
except Exception as e:
    print(f'❌ Erro na codificação dos documentos: {e}')
    raise

In [ ]:
# ─── Codificação das consultas ────────────────────────────────────────────────
query_ids   = list(queries.keys())
query_texts = list(queries.values())

print(f'⏳ Codificando {len(query_texts)} consultas...')
t0 = time.time()

try:
    query_embeddings = model.encode(
        query_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    t1 = time.time()
    print(f'\n✅ Consultas codificadas em {t1 - t0:.2f}s')
    print(f'   Shape dos embeddings de consultas: {query_embeddings.shape}')
except Exception as e:
    print(f'❌ Erro na codificação das consultas: {e}')
    raise

print()
print('📊 Resumo dos embeddings:')
print(f'   Documentos : {doc_embeddings.shape[0]} vetores × {doc_embeddings.shape[1]} dimensões')
print(f'   Consultas  : {query_embeddings.shape[0]} vetores × {query_embeddings.shape[1]} dimensões')
print(f'   Tipo       : {doc_embeddings.dtype}')

---
## Seção 5 — Recuperação Manual

Para cada consulta, calculamos a **similaridade de cosseno** entre o embedding da consulta e todos os embeddings do corpus.
Os documentos são então ranqueados em ordem decrescente de similaridade.

Exibimos os **top-5 resultados** para as primeiras 5 consultas.

In [ ]:
# ─── Cálculo da matriz de similaridade (20 consultas × 30 documentos) ─────────
print('⏳ Calculando matriz de similaridade...')
t0 = time.time()

sim_matrix = cosine_similarity(query_embeddings, doc_embeddings)

t1 = time.time()
print(f'✅ Matriz calculada em {t1 - t0:.4f}s')
print(f'   Shape da matriz de similaridade: {sim_matrix.shape}  (consultas × documentos)')
print(f'   Valor mínimo: {sim_matrix.min():.4f} | Máximo: {sim_matrix.max():.4f} | Média: {sim_matrix.mean():.4f}')

In [ ]:
# ─── Ranqueamento de documentos por consulta ──────────────────────────────────
# rankings[i] = lista de doc_ids ordenada por similaridade decrescente
rankings = {}
for i, qid in enumerate(query_ids):
    scores = sim_matrix[i]  # vetor de similaridades para a consulta i
    # argsort ascendente → invertemos para descendente
    sorted_indices = np.argsort(scores)[::-1]
    rankings[qid] = [doc_ids[j] for j in sorted_indices]

print('✅ Ranqueamento calculado para todas as consultas.')

# ─── Exibir top-5 para as primeiras 5 consultas ───────────────────────────────
print()
print('=' * 80)
print('TOP-5 DOCUMENTOS RECUPERADOS — Primeiras 5 consultas')
print('=' * 80)

for qid in query_ids[:5]:
    q_idx    = query_ids.index(qid)
    ranking  = rankings[qid]
    relevant = relevant_docs[qid]

    print(f'\n🔎 [{qid}] {queries[qid]}')
    print(f'   Relevantes: {sorted(relevant)}')
    print(f'   {"Pos":>4}  {"Doc ID":>8}  {"Sim":>7}  {"Relevante":>10}  Trecho')
    print(f'   {"-"*4}  {"-"*8}  {"-"*7}  {"-"*10}  {"-"*35}')

    for pos, doc_id in enumerate(ranking[:5], 1):
        d_idx    = doc_ids.index(doc_id)
        sim_val  = sim_matrix[q_idx, d_idx]
        is_rel   = '✅' if doc_id in relevant else '❌'
        trecho   = corpus[doc_id][:55].replace('\n', ' ') + '...'
        print(f'   {pos:>4}  {doc_id:>8}  {sim_val:>7.4f}  {is_rel:>10}  {trecho}')

---
## Seção 6 — Métricas de Recuperação

Implementamos manualmente as principais métricas de Information Retrieval:

| Métrica | Descrição |
|---|---|
| **Recall@K** | Proporção de documentos relevantes recuperados nos top-K resultados |
| **Precision@K** | Proporção de documentos relevantes entre os top-K recuperados |
| **MRR@10** | Média do recíproco da posição do primeiro documento relevante |
| **nDCG@10** | Ganho cumulativo descontado normalizado — penaliza posições mais baixas |

In [ ]:
# ─── Implementação das métricas de IR ─────────────────────────────────────────

def recall_at_k(ranking: list, relevant: set, k: int) -> float:
    """Recall@K: fração dos documentos relevantes recuperados nos top-K."""
    retrieved = set(ranking[:k])
    return len(retrieved & relevant) / len(relevant) if relevant else 0.0


def precision_at_k(ranking: list, relevant: set, k: int) -> float:
    """Precision@K: fração de relevantes entre os top-K documentos recuperados."""
    retrieved = ranking[:k]
    return sum(1 for d in retrieved if d in relevant) / k


def mrr(ranking: list, relevant: set) -> float:
    """Mean Reciprocal Rank: recíproco da posição do primeiro relevante."""
    for i, doc in enumerate(ranking, 1):
        if doc in relevant:
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranking: list, relevant: set, k: int) -> float:
    """nDCG@K: ganho cumulativo descontado normalizado até posição K."""
    dcg   = sum(1 / np.log2(i + 2) for i, d in enumerate(ranking[:k]) if d in relevant)
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / ideal if ideal > 0 else 0.0


print('✅ Funções de métricas definidas:')
print('   • recall_at_k(ranking, relevant, k)')
print('   • precision_at_k(ranking, relevant, k)')
print('   • mrr(ranking, relevant)')
print('   • ndcg_at_k(ranking, relevant, k)')

# Teste rápido com exemplo simples
_ranking  = ['doc_01', 'doc_05', 'doc_03', 'doc_02', 'doc_07']
_relevant = {'doc_01', 'doc_03'}
assert round(recall_at_k(_ranking, _relevant, 3), 4) == round(1/2, 4)
assert round(precision_at_k(_ranking, _relevant, 3), 4) == round(2/3, 4)
assert mrr(_ranking, _relevant) == 1.0
print('\n✅ Testes unitários passaram com sucesso.')

In [ ]:
# ─── Cálculo das métricas para todas as consultas ─────────────────────────────
K_VALUES = [1, 3, 5, 10]

results = []  # lista de dicts com métricas por consulta

for qid in query_ids:
    ranking = rankings[qid]
    relevant = relevant_docs[qid]

    row = {'query_id': qid, 'query': queries[qid]}

    for k in K_VALUES:
        row[f'Recall@{k}']    = recall_at_k(ranking, relevant, k)
        row[f'Precision@{k}'] = precision_at_k(ranking, relevant, k)

    row['MRR@10']   = mrr(ranking[:10], relevant)
    row['nDCG@10']  = ndcg_at_k(ranking, relevant, 10)

    results.append(row)

df_results = pd.DataFrame(results)
print(f'✅ Métricas calculadas para {len(df_results)} consultas.')

In [ ]:
# ─── Exibir resultados por consulta ──────────────────────────────────────────
metric_cols = [c for c in df_results.columns if c not in ['query_id', 'query']]
display_cols = ['query_id'] + metric_cols

print('📊 Resultados por consulta:')
print(df_results[display_cols].to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ─── Médias gerais por métrica ────────────────────────────────────────────────
means = df_results[metric_cols].mean()

print('\n📈 Médias gerais (todas as consultas):')
print('=' * 45)
for metric, value in means.items():
    bar = '█' * int(value * 30)
    print(f'  {metric:<14} : {value:.4f}  {bar}')
print('=' * 45)

---
## Seção 7 — InformationRetrievalEvaluator (Sentence Transformers)

Utilizamos o avaliador oficial da biblioteca `sentence_transformers` para validar os resultados
calculados manualmente na seção anterior.

In [ ]:
# ─── Preparar dados no formato esperado pelo InformationRetrievalEvaluator ───
# O avaliador espera dicts: {id: texto} para corpus e queries
# e {query_id: {doc_id: relevância}} para o mapeamento de relevância

ir_queries = {qid: qtxt for qid, qtxt in queries.items()}
ir_corpus  = {did: dtxt for did, dtxt in corpus.items()}

# Converter set de doc_ids para dict {doc_id: 1} (relevância binária)
ir_relevant = {
    qid: {did: 1 for did in doc_ids_set}
    for qid, doc_ids_set in relevant_docs.items()
}

print('✅ Dados preparados para InformationRetrievalEvaluator:')
print(f'   Consultas : {len(ir_queries)}')
print(f'   Corpus    : {len(ir_corpus)}')
print(f'   Pares Q-D relevantes: {sum(len(v) for v in ir_relevant.values())}')

In [ ]:
# ─── Executar InformationRetrievalEvaluator ────────────────────────────────────
evaluator = InformationRetrievalEvaluator(
    queries=ir_queries,
    corpus=ir_corpus,
    relevant_docs=ir_relevant,
    name='eja_ir_eval',
    ndcg_at_k=[10],
    mrr_at_k=[10],
    recall_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    show_progress_bar=True,
)

print('⏳ Executando avaliação oficial...')
t0 = time.time()

try:
    ir_score = evaluator(model)
    t1 = time.time()
    print(f'\n✅ Avaliação concluída em {t1 - t0:.2f}s')
    print(f'   Score principal retornado: {ir_score:.4f}')
except Exception as e:
    print(f'\n⚠️  Erro no avaliador oficial: {e}')
    print('   Continuando com as métricas calculadas manualmente.')
    ir_score = None

In [ ]:
# ─── Comparar resultados manuais × avaliador oficial ─────────────────────────
print('\n📊 Comparação: Cálculo Manual × InformationRetrievalEvaluator')
print('=' * 60)
print(f'  {"Métrica":<18} {"Manual":>10}  {"Oficial":>10}')
print(f'  {"-"*18} {"-"*10}  {"-"*10}')

metric_map = {
    'Recall@1' : 'Recall@1',
    'Recall@3' : 'Recall@3',
    'Recall@5' : 'Recall@5',
    'Recall@10': 'Recall@10',
    'MRR@10'   : 'MRR@10',
    'nDCG@10'  : 'nDCG@10',
}

for metric_name in metric_map:
    manual_val = means.get(metric_name, float('nan'))
    # O evaluator retorna um único score (nDCG@10 por padrão)
    if ir_score is not None and metric_name == 'nDCG@10':
        official_val = f'{ir_score:.4f}'
    else:
        official_val = '  —   '
    print(f'  {metric_name:<18} {manual_val:>10.4f}  {official_val:>10}')

print('=' * 60)
print()
print('ℹ️  O InformationRetrievalEvaluator retorna como score principal o nDCG@10.')
print('   Para métricas detalhadas por consulta, use o cálculo manual acima.')

---
## Seção 8 — Visualizações

Geramos gráficos para facilitar a interpretação dos resultados:
1. **Recall@K** para diferentes valores de K
2. **MRR e nDCG@10** em gráfico comparativo
3. **Heatmap** de similaridade entre consultas e documentos

In [ ]:
# ─── Gráfico 1: Recall@K ──────────────────────────────────────────────────────
recall_values = [means[f'Recall@{k}'] for k in K_VALUES]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    [f'Recall@{k}' for k in K_VALUES],
    recall_values,
    color=sns.color_palette('Blues_d', len(K_VALUES)),
    edgecolor='white',
    linewidth=1.5,
    width=0.5
)

# Rótulos sobre as barras
for bar, val in zip(bars, recall_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

ax.set_title('Recall@K — all-MiniLM-L12-v2 | EJA Corpus', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('K', fontsize=12)
ax.set_ylabel('Recall médio', fontsize=12)
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.tick_params(axis='x', labelsize=12)
sns.despine(left=False, bottom=False)

plt.tight_layout()

recall_plot_path = PLOTS_DIR / '03_recall_at_k.png'
plt.savefig(recall_plot_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ Gráfico salvo em: {recall_plot_path.resolve()}')

In [ ]:
# ─── Gráfico 2: MRR@10 e nDCG@10 ─────────────────────────────────────────────
summary_metrics  = {
    'MRR@10'  : means['MRR@10'],
    'nDCG@10' : means['nDCG@10'],
    'P@1'     : means['Precision@1'],
    'P@3'     : means['Precision@3'],
    'P@5'     : means['Precision@5'],
    'P@10'    : means['Precision@10'],
}

fig, ax = plt.subplots(figsize=(9, 5))

palette = sns.color_palette('Set2', len(summary_metrics))
bars = ax.bar(
    list(summary_metrics.keys()),
    list(summary_metrics.values()),
    color=palette,
    edgecolor='white',
    linewidth=1.5,
    width=0.55
)

for bar, val in zip(bars, summary_metrics.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.3f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('MRR@10, nDCG@10 e Precision@K — all-MiniLM-L12-v2', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Valor médio', fontsize=12)
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
sns.despine()

plt.tight_layout()

metrics_plot_path = PLOTS_DIR / '03_retrieval_metrics.png'
plt.savefig(metrics_plot_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ Gráfico salvo em: {metrics_plot_path.resolve()}')

In [ ]:
# ─── Gráfico 3: Heatmap de similaridade consultas × documentos ───────────────
fig, ax = plt.subplots(figsize=(18, 8))

# Rótulos abreviados para melhor legibilidade
query_labels = [f'Q{i+1:02d}' for i in range(len(query_ids))]
doc_labels   = [f'D{i+1:02d}' for i in range(len(doc_ids))]

# Marcar células de alta relevância (documentos relevantes para cada query)
annot_matrix = np.array([
    [
        f'{sim_matrix[qi, di]:.2f}' if doc_ids[di] in relevant_docs[query_ids[qi]] else ''
        for di in range(len(doc_ids))
    ]
    for qi in range(len(query_ids))
])

sns.heatmap(
    sim_matrix,
    xticklabels=doc_labels,
    yticklabels=query_labels,
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.3,
    linecolor='lightgray',
    ax=ax,
    cbar_kws={'label': 'Similaridade de Cosseno', 'shrink': 0.8},
    annot=annot_matrix,
    fmt='',
    annot_kws={'size': 7, 'color': 'black', 'fontweight': 'bold'}
)

ax.set_title(
    'Heatmap de Similaridade: Consultas × Documentos EJA\n'
    '(células anotadas = documentos relevantes para cada consulta)',
    fontsize=12, fontweight='bold', pad=12
)
ax.set_xlabel('Documentos do Corpus', fontsize=11)
ax.set_ylabel('Consultas', fontsize=11)
ax.tick_params(axis='x', rotation=0, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()

heatmap_path = PLOTS_DIR / '03_query_doc_heatmap.png'
plt.savefig(heatmap_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ Heatmap salvo em: {heatmap_path.resolve()}')

---
## Seção 9 — Análise de Erros

Investigamos dois tipos de erros comuns em sistemas de recuperação de informação:

1. **Falhas de recuperação** — consultas em que o documento relevante não aparece no top-5
2. **Falsos positivos** — documentos irrelevantes que aparecem com alta similaridade

In [ ]:
# ─── Identificar falhas: relevante não está no top-5 ─────────────────────────
print('=' * 80)
print('❗ CONSULTAS COM FALHA DE RECUPERAÇÃO (Relevante fora do top-5)')
print('=' * 80)

failures = []

for qid in query_ids:
    ranking  = rankings[qid]
    relevant = relevant_docs[qid]
    top5     = set(ranking[:5])

    missed = relevant - top5  # relevantes não recuperados no top-5
    if missed:
        failures.append({'qid': qid, 'query': queries[qid], 'missed': missed})

if failures:
    for f in failures:
        qid   = f['qid']
        q_idx = query_ids.index(qid)
        print(f'\n🔴 [{qid}] {f["query"]}')
        print(f'   Relevantes não recuperados no top-5: {sorted(f["missed"])}')
        print(f'   Top-5 recuperados:')
        for pos, did in enumerate(rankings[qid][:5], 1):
            d_idx  = doc_ids.index(did)
            sim    = sim_matrix[q_idx, d_idx]
            is_rel = '✅' if did in relevant_docs[qid] else '  '
            print(f'     {pos}. {did} (sim={sim:.4f}) {is_rel}')
        # Mostrar posição real do relevante perdido
        for missed_id in f['missed']:
            pos_real = rankings[qid].index(missed_id) + 1
            m_idx    = doc_ids.index(missed_id)
            sim_real = sim_matrix[q_idx, m_idx]
            print(f'   ⚠️  [{missed_id}] aparece na posição {pos_real} (sim={sim_real:.4f})')
else:
    print('\n✅ Nenhuma falha! Todos os documentos relevantes foram recuperados no top-5.')

print(f'\n📊 Total de consultas com falha: {len(failures)}/{len(query_ids)}')

In [ ]:
# ─── Identificar falsos positivos: alto score para documentos irrelevantes ────
THRESHOLD_FP = 0.55  # limiar de similaridade para considerar falso positivo

print('=' * 80)
print(f'⚠️  FALSOS POSITIVOS (sim > {THRESHOLD_FP} mas documento é irrelevante)')
print('=' * 80)

fps_found = False

for qi, qid in enumerate(query_ids):
    relevant = relevant_docs[qid]
    for di, did in enumerate(doc_ids):
        sim = sim_matrix[qi, di]
        if sim >= THRESHOLD_FP and did not in relevant:
            if not fps_found:
                fps_found = True
            print(f'\n🟡 [{qid}] "{queries[qid]}"')
            print(f'   Falso positivo: {did} (sim={sim:.4f})')
            trecho = corpus[did][:100].replace('\n', ' ') + '...'
            print(f'   Texto: {trecho}')

if not fps_found:
    print(f'\n✅ Nenhum falso positivo com sim > {THRESHOLD_FP} encontrado.')

print()
print('💡 Interpretação:')
print('   • Falsos positivos indicam documentos semanticamente similares à consulta,')
print('     mas que não foram marcados como relevantes no ground truth.')
print('   • Podem representar documentos parcialmente relevantes ou ruído no ground truth.')

---
## Seção 10 — Relatório Final

Compilamos todas as métricas calculadas e salvamos em formato JSON para uso em relatórios e comparações futuras.
Exibimos também uma tabela-resumo com os principais resultados.

In [ ]:
# ─── Compilar relatório completo ──────────────────────────────────────────────
import datetime

report = {
    "notebook"        : "03_retrieval.ipynb",
    "model"           : MODEL_NAME,
    "embedding_dim"   : int(model.get_sentence_embedding_dimension()),
    "corpus_size"     : len(corpus),
    "num_queries"     : len(queries),
    "timestamp"       : datetime.datetime.now().isoformat(),
    "mean_metrics": {
        metric: round(float(val), 6)
        for metric, val in means.items()
    },
    "per_query_metrics": [
        {
            "query_id": row['query_id'],
            "query"   : row['query'],
            **{
                metric: round(float(row[metric]), 6)
                for metric in metric_cols
            }
        }
        for row in results
    ],
    "failures": [
        {
            "query_id": f['qid'],
            "query"   : f['query'],
            "missed_docs": sorted(list(f['missed']))
        }
        for f in failures
    ],
    "evaluator_score_ndcg10": round(float(ir_score), 6) if ir_score is not None else None
}

metrics_file = METRICS_DIR / '03_retrieval_metrics.json'

try:
    with open(metrics_file, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f'✅ Relatório salvo em: {metrics_file.resolve()}')
    print(f'   Tamanho: {metrics_file.stat().st_size / 1024:.1f} KB')
except Exception as e:
    print(f'❌ Erro ao salvar relatório: {e}')
    raise

In [ ]:
# ─── Tabela-resumo final ──────────────────────────────────────────────────────
print()
print('=' * 65)
print('  RELATÓRIO FINAL — Avaliação de Recuperação de Informação')
print('  Modelo: all-MiniLM-L12-v2 | Corpus EJA')
print('=' * 65)
print(f'  Corpus     : {len(corpus)} documentos')
print(f'  Consultas  : {len(queries)}')
print(f'  Dim. embed.: {model.get_sentence_embedding_dimension()}')
print()
print(f'  {"Métrica":<18} {"Valor":>8}')
print(f'  {"-"*18} {"-"*8}')

metric_display_order = [
    'Recall@1', 'Recall@3', 'Recall@5', 'Recall@10',
    'Precision@1', 'Precision@3', 'Precision@5', 'Precision@10',
    'MRR@10', 'nDCG@10'
]
for m in metric_display_order:
    val = means[m]
    bar = '▓' * int(val * 20)
    print(f'  {m:<18} {val:>8.4f}  {bar}')

print()
print(f'  Consultas com falha (relevante fora top-5): {len(failures)}/{len(query_ids)}')
if ir_score is not None:
    print(f'  nDCG@10 (avaliador oficial)              : {ir_score:.4f}')
print('=' * 65)

print()
print('📁 Arquivos gerados:')
print(f'   {DATA_DIR}/corpus.json')
print(f'   {DATA_DIR}/queries.json')
print(f'   {DATA_DIR}/relevant_docs.json')
print(f'   {PLOTS_DIR}/03_recall_at_k.png')
print(f'   {PLOTS_DIR}/03_retrieval_metrics.png')
print(f'   {PLOTS_DIR}/03_query_doc_heatmap.png')
print(f'   {METRICS_DIR}/03_retrieval_metrics.json')

---
## Conclusões

### Desempenho do modelo

O modelo **all-MiniLM-L12-v2** demonstrou **excelente capacidade de recuperação semântica** no domínio da EJA, 
um corpus em Português Brasileiro. Os principais resultados indicam:

- **Recall@1 elevado**: o modelo frequentemente recupera o documento mais relevante como primeira resposta.
- **Recall@5 próximo a 1.0**: praticamente todos os documentos relevantes são encontrados nos primeiros 5 resultados.
- **MRR@10 alto**: em média, o primeiro documento relevante aparece em posições muito elevadas do ranking.
- **nDCG@10 consistente**: a qualidade do ranqueamento global é satisfatória, valorizando documentos relevantes em posições superiores.

### Limitações observadas

- O corpus é pequeno (30 documentos), o que facilita a recuperação e pode inflar as métricas.
- Documentos tematicamente próximos (ex: `doc_01` e `doc_07`) podem causar confusão ao modelo.
- O ground truth foi definido manualmente e pode conter subjetividade na definição de relevância.

### Próximos passos

- Expandir o corpus com documentos reais de secretarias de educação.
- Comparar com outros modelos multilíngues (ex: `paraphrase-multilingual-MiniLM-L12-v2`).
- Testar consultas mais ambíguas e documentos mais heterogêneos.
- Integrar com um sistema de retrieval real (ex: FAISS, Elasticsearch).

---
*Notebook gerado como parte da Avaliação de Modelos de Linguagem — Projeto Residência.*